# Scratchpad

A place to run things live. Nothing here is graded and nothing here can break
the exercises.

**Run the first cell before you start.** It connects to the School's model
server and tells you if it cannot.

Everything below is grouped by block. Each group is self-contained, so you can
jump straight to the one you need without running the ones above it — except
retrieval, which needs its index cell run once.

## Connect

In [2]:
from agentic_fdp import chat_model, embeddings, chat_base_url, embed_base_url
from agentic_fdp.live import show, rule, ask, run_tools, hits

model = chat_model()

print("chat      ", chat_base_url())
try:
    print("embeddings", embed_base_url())
except Exception as e:
    print("embeddings  NOT REACHABLE —", e)

show(ask(model, "Reply with exactly: ready", quiet=True))

chat       http://172.17.16.12:8001/v1
embeddings http://172.17.16.12:8002/v1
ready


---
## Warm-up — one call, and the system message

The smallest thing worth showing: same question, different instruction.

In [3]:
QUESTION = "In two sentences, what is an AI agent?"

rule("no system message")
ask(model, QUESTION)

rule("as a strict examiner")
ask(model, QUESTION, "You are a strict examiner. One line, no examples.")

rule("as an encouraging tutor")
ask(model, QUESTION, "You are an encouraging tutor. Use one everyday example.")


────────────────────────────────────────────────────────────────────────────────────────
no system message
────────────────────────────────────────────────────────────────────────────────────────
An AI agent is an autonomous system that uses artificial intelligence to perceive its
environment, reason through complex tasks, and take actions to achieve specific goals.
Unlike standard chatbots that merely respond to prompts, an agent can independently use
tools, interact with other software, and make decisions to complete multi-step
workflows.

[0.9s]

────────────────────────────────────────────────────────────────────────────────────────
as a strict examiner
────────────────────────────────────────────────────────────────────────────────────────
An AI agent is an autonomous system designed to perceive its environment and take
actions to achieve specific goals. It utilizes reasoning and decision-making
capabilities to interact with tools and complete complex tasks independently.

[0.5s]

"An AI agent is a smart program that doesn't just answer questions, but actually takes actions to complete specific goals for you. Think of it like a digital personal assistant: while a standard AI is like a cookbook that gives you recipes, an AI agent is like a chef who actually goes into the kitchen to cook the meal!"

---
## Block 1 — Tool Use

`run_tools` is the loop participants write, with printing added. Change the
question and watch the sequence change.

In [ ]:
from exercises.ex01_tool_use.tools import ALL_TOOLS, calculator, course_credits

run_tools(model, "How many credits do 23AID304 and 24AIM332 come to together?", ALL_TOOLS)

In [ ]:
# No tool needed — watch it call nothing.
run_tools(model, "Which of 23AID311 and 23MAT216 is worth more?", ALL_TOOLS)

In [ ]:
# Define a tool live. The docstring is what the model reads when deciding.
from langchain_core.tools import tool

@tool
def room_temperature(building: str) -> str:
    """Get the current temperature in a named building on campus."""
    return f"It is 24 degrees in {building}."

run_tools(model, "Is it warm in the CEN building?", [room_temperature])

---
## Block 2 — Chaining and Routing

In [ ]:
TEXT = """
The Amrita School of AI runs a SLURM cluster called asaicompute. It has three
nodes with six GPUs between them, shared home directories over GlusterFS, and
most interactive work arrives through a web portal rather than over SSH.
"""

rule("stage 1 — extract")
facts = ask(model, TEXT, "Extract the key facts as short bullet points.")

rule("stage 2 — reads stage 1's output, not the original")
ask(model, facts, "Write one headline of at most twelve words. Headline only.")

In [ ]:
# Naming the categories is not enough — describe them. A bare list of three
# words sends almost everything to 'other'.
ROUTER = (
    "Classify the question into exactly one category.\n"
    "Reply with one word only, no punctuation, no explanation.\n\n"
    "maths  — arithmetic, algebra, calculation, anything with numbers\n"
    "policy — course credits, regulations, attendance, examination rules\n"
    "other  — anything else\n"
)

for q in ["If a job needs 4 GPUs and each node has 2, how many nodes?",
          "How many credits does 23AID304 carry?",
          "What should I read to learn SLURM?"]:
    label = ask(model, q, ROUTER, quiet=True).strip().strip(".").lower()
    print(f"{label:>8}  <-  {q}")

---
## Block 3 — Reflection

In [ ]:
BRIEF = ("Explain gradient descent to someone who has never studied calculus. "
         "Exactly 40 words. Must use the word 'hill'. Must not use: learn, "
         "model, data, train, loss, minimum, function, algorithm.")

rule("draft")
draft = ask(model, BRIEF, "You are a technical writer. Write the requested text.")

rule("critique")
critique = ask(model, f"BRIEF:\n{BRIEF}\n\nTEXT:\n{draft}",
               "You are a strict reviewer. If it fully meets the brief reply "
               "exactly APPROVED, else list what is wrong as bullet points.")

In [ ]:
rule("revision")
ask(model, f"BRIEF:\n{BRIEF}\n\nTEXT:\n{draft}\n\nCRITIQUE:\n{critique}",
    "Rewrite the text so it addresses every point of the critique. "
    "Reply with the revised text only.")

---
## Block 4 — Retrieval

**Run the index cell once.** It takes a few seconds; everything after it is
instant.

This section runs *your* block 4 code, so it needs that block finished. That is
deliberate: the scratchpad is for experimenting with what you built, and
inlining a working version here would just be handing you the answer.

In [ ]:
from exercises.ex04_rag.agent import build_index, load_corpus, split

try:
    chunks = split(load_corpus())
    print(f"{len(chunks)} chunks")
    index = build_index(chunks, embeddings())
    print("indexed")
except NotImplementedError as e:
    index = None
    print(f"Block 4 is not finished yet — {e}")
    print("Fill in the TODOs in exercises/ex04_rag/agent.py, then re-run this cell.")

In [ ]:
q = "What is the scope of the M.Tech Data Science programme?"

assert index is not None, "Run the index cell above first (and finish block 4)."
found = index.similarity_search(q, k=4)
hits(found)

context = "\n\n".join(f"[{c.metadata['source']}]\n{c.page_content}" for c in found)

rule("answer")
ask(model, f"CONTEXT:\n{context}\n\nQUESTION: {q}",
    "Answer using only the context. If it is not there, say exactly: "
    "I cannot find that in the documents. Cite the source file for each fact.")

In [ ]:
# Ask something the documents do not cover, and watch it refuse.
q = "What is the hostel mess fee?"

assert index is not None, "Run the index cell above first (and finish block 4)."
found = index.similarity_search(q, k=4)
hits(found)

context = "\n\n".join(f"[{c.metadata['source']}]\n{c.page_content}" for c in found)

rule("answer")
ask(model, f"CONTEXT:\n{context}\n\nQUESTION: {q}",
    "Answer using only the context. If it is not there, say exactly: "
    "I cannot find that in the documents.")

---
## Block 5 — Multi-Agent

Streams the graph so each supervisor decision appears as it is made. Runs your
block 5 code, so finish that block first.

In [ ]:
from exercises.ex05_capstone.agent import build_graph

question = "What does the Machine Learning course cover?"

try:
    graph = build_graph(model)
    for event in graph.stream({"question": question, "notes": [], "draft": "",
                               "steps": 0, "next": ""}):
        for node, update in event.items():
            if node == "supervisor":
                print(f"supervisor (step {update['steps']}): {update['next']}")
            elif node == "research":
                print(f"researcher: {' '.join(update['notes'][0].split())[:120]}...")
            elif node == "write":
                rule("draft")
                show(update["draft"])
except NotImplementedError as e:
    print(f"Block 5 is not finished yet — {e}")

---
## Spare

Room to improvise. `model`, `index`, `ALL_TOOLS`, `show`, `rule`, `ask`,
`run_tools` and `hits` are all still in scope.